In [10]:
def clean_text(text: str) -> str:
    """Remove newlines, tabs, extra spaces, punctuation and lowercase."""
    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[!?]+", "", text)
    return text.strip().lower()

## SQLite

In [2]:

import sqlite3

conn = sqlite3.connect(r"/src/triplet_extraction/batch_preprocessing/GTVT_law.db")
cursor = conn.cursor()

def extract_all_with_parents(cursor):
    # Step 1: Find all "Điểm" nodes
    cursor.execute("SELECT * FROM laws WHERE title LIKE '%Điểm%'")
    rows = cursor.fetchall()
    if not rows:
        return []

    columns = [col[0] for col in cursor.description]
    results = []

    # Step 2: For each node, find its parent chain
    for row in rows:
        node = dict(zip(columns, row))
        parent_chain = []

        current = node
        visited = set()

        while current['parent_id'] is not None and current['parent_id'] not in visited:
            visited.add(current['id'])
            cursor.execute("SELECT * FROM laws WHERE id = ?", (current['parent_id'],))
            parent = cursor.fetchone()
            if not parent:
                break
            parent_dict = dict(zip(columns, parent))
            parent_chain.append(parent_dict)
            current = parent_dict

        results.append({
            "target": node,
            "ancestors": parent_chain[::-1]
        })

    return results

In [4]:
diem_db_data = extract_all_with_parents(cursor)
diem_data = {}

for diem in diem_db_data:
    content = ""
    title = ""

    for r in diem["ancestors"]:
        title += r['title'] + " "
        if not (r['title'].strip().startswith("Chương") or r['title'].strip().startswith("Điều")):
            content += r['content'] + "\n"

    title += diem['target']["title"]
    content += diem['target']["content"]

    so_hieu = diem['target']['so_hieu']
    ID = diem['target']['id']

    diem_data[ID] = {
        "so_hieu": so_hieu,
        "title": title,
        "content": content
    }

## MongoDB

In [1]:
from pymongo import MongoClient
import re

# Connect to MongoDB
client = MongoClient('mongodb://localhost:27017/')
db = client['KB_PROPERTY_LAW']
collection = db['legal_sections']

In [2]:
KEEP_SO_HIEU = [
    "31/2024/QH15",
    "101/2024/NĐ-CP",
    "102/2024/NĐ-CP",
    "103/2024/NĐ-CP",
    "226/2025/NĐ-CP",
    "27/2023/QH15",
    "95/2024/NĐ-CP",
    "29/2023/QH15",
    "96/2024/NĐ-CP",
    "91/2015/QH13",
    "52/2014/QH13"
]

In [3]:
SIMPLIFY_SYSTEM_PROMPT = """
Role: You are a professional Legal and Linguistics AI Assistant specializing in Vietnamese law. Your task is to deconstruct complex legal texts into independent, direct simple sentences.
Objective: Transform legal clauses into standalone simple sentences. Each sentence must be a direct legal statement, avoiding introductory fillers or explanatory bridges.
Strict "Directness" Rules:
No Filler Subjects: Do not use phrases like "Một loại hành vi là..." (One type of behavior is...), "Bao gồm các loại..." (Includes types of...), or "Được xác định như sau" (Is defined as follows).
Direct Predication: Connect the main Subject directly to the specific Action or Object.
Incorrect: "Một loại hành vi vi phạm là làm sai lệch hồ sơ."
Correct: "Hành vi vi phạm quy định về hồ sơ bao gồm hành vi làm sai lệch hồ sơ." (Or simply: "Làm sai lệch hồ sơ là hành vi vi phạm quy định về hồ sơ.")
The "Simple Sentence" Constraint:
Exactly one Subject and one Predicate.
No conjunctions (và, hoặc, nhưng, mà, còn, rồi...).
No commas (,) or semicolons (;) to link clauses.
Copy Forward Context: If the input lists sub-items under a heading, integrate the full heading context into every single sub-item to ensure they are legally complete.
Output Format:
Return strictly a JSON object.
If information is insufficient: {"need_more_information": "..."}.
If successful: {"simplified_sentences": ["Direct Sentence 1.", "Direct Sentence 2."]}.
Example Transformation: Input: "Hành vi vi phạm quy định về hồ sơ địa giới bao gồm: (a) Làm sai lệch sơ đồ; (b) Làm sai lệch bảng tọa độ." Output: { "simplified_sentences": [ "Hành vi làm sai lệch sơ đồ là hành vi vi phạm quy định về hồ sơ địa giới.", "Hành vi làm sai lệch bảng tọa độ là hành vi vi phạm quy định về hồ sơ địa giới." ] }
"""

In [4]:
# Load only what we need
docs = list(db.legal_sections.find({}))

# Collect all parent_ids that appear
parent_ids = {
    doc["parent_id"]
    for doc in docs
    if doc.get("parent_id") is not None
}

# Leaf nodes = ids that never appear as a parent_id
leaf_nodes = [
    doc for doc in docs
    if doc["_id"] not in parent_ids
]

print(f"Leaf nodes: {len(leaf_nodes)}")

all_docs = list(db.legal_sections.find({}))

id_map = {doc["_id"]: doc for doc in all_docs}

def collect_path(leaf):
    path = []
    current = leaf

    while current:
        path.append(current)
        pid = current.get("parent_id")
        current = id_map.get(pid)

    # reverse to get root → leaf
    path.reverse()
    return path

sections_data = []

for leaf in leaf_nodes:
    path_nodes = collect_path(leaf)
    if leaf["so_hieu"] not in KEEP_SO_HIEU:
        continue

    if leaf.get("is_phu_luc"):
        continue

    if leaf.get("is_amendment"):
        continue

    result = {
        "leaf_id": leaf["_id"],
        "so_hieu": leaf.get("so_hieu"),
        "full_path": leaf.get("full_path"),
        "combined_content": "\n".join(
            n["content"]
            for n in path_nodes
            if n.get("content") and n["type"] in ["điều", "khoản", "điểm"]
        )
    }

    if not result["combined_content"]:
        continue

    sections_data.append(result)
print(f"Total sections collected: {len(sections_data)}")

Leaf nodes: 11670
Total sections collected: 8686


In [5]:
import random

random_section = random.choice(sections_data)

print("ID:", random_section["leaf_id"])
print(f"Full Path: {random_section['full_path']}")
print(random_section["combined_content"])

ID: 7bf064aad24783973644892f7b6494d727ec602dc90d91e929d53a5ca6274b92
Full Path: 101/2024/NĐ-CP_chương iv_điều 52_khoản 7
xây dựng, cập nhật và quản lý cơ sở dữ liệu quốc gia về đất đai
Việc cập nhật cơ sở dữ liệu quốc gia về đất đai phải được thực hiện thường xuyên, đảm bảo tính pháp lý, chính xác, tính duy nhất của đối tượng, đầy đủ, kịp thời trong quá trình giải quyết thủ tục hành chính về đất đai thông qua phần mềm của Hệ thống thông tin quốc gia về đất đai; đối với các trường hợp có thay đổi, biến động về thông tin, dữ liệu được cơ quan có thẩm quyền phê duyệt mà không gắn với việc thực hiện thủ tục hành chính về đất đai của người sử dụng đất, chủ sở hữu tài sản gắn liền với đất thì phải cập nhật, chỉnh lý, bổ sung vào cơ sở dữ liệu quốc gia về đất đai.


## Create batch api file

In [38]:
from openai import OpenAI
from dotenv import load_dotenv
import os

# Load .env files
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

def get_gpt_response(user_prompt, system_prompt, api_key, model="gpt-4o"):
    client = OpenAI(api_key=api_key)
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
    )
    print(response.choices[0].message.content)
    return response.choices[0].message.content

In [8]:
import tiktoken

encoding = tiktoken.get_encoding("cl100k_base")

def count_chat_tokens(messages):
    """
    Rough but safe token count for chat.completions
    """
    tokens = 0
    for msg in messages:
        tokens += 4  # role + formatting overhead
        tokens += len(encoding.encode(msg["content"]))
    tokens += 2  # assistant reply priming
    return tokens

In [11]:
import json
import tiktoken
from typing import List

MODEL_NAME = "gpt-4.1-mini"
MAX_FILE_TOKENS = 2_000_000
MAX_TASK_TOKENS = 500_000   # safety limit per task before splitting content
OUTPUT_PREFIX = "batch_property_law_v2"

def normalize_for_json(obj):
    if isinstance(obj, dict):
        return {k: normalize_for_json(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [normalize_for_json(v) for v in obj]
    elif isinstance(obj, tuple):
        return list(obj)
    return obj


def split_text_by_tokens(text: str, max_tokens: int) -> List[str]:
    """
    Split long content into token-safe chunks
    """
    tokens = encoding.encode(text)
    chunks = []

    for i in range(0, len(tokens), max_tokens):
        chunk_tokens = tokens[i:i + max_tokens]
        chunks.append(encoding.decode(chunk_tokens))

    return chunks

tasks = []
for section in sections_data:
    so_hieu = section["so_hieu"]
    section_id = section["leaf_id"]
    full_path = section["full_path"]
    content = clean_text(section["combined_content"])

    # split oversized content first
    content_chunks = split_text_by_tokens(content, MAX_TASK_TOKENS)

    if not section_id:
        print("No section id error")
        break

    for idx, chunk in enumerate(content_chunks, start=1):
        task = {
            "custom_id": section_id,
            "method": "POST",
            "url": "/v1/chat/completions",
            "body": {
                "model": MODEL_NAME,
                "messages": [
                    {"role": "system", "content": SIMPLIFY_SYSTEM_PROMPT},
                    {"role": "user", "content": f'Sentence need simplify:  "{chunk}"'},
                ],
            },
        }

        tokens = count_chat_tokens(task["body"]["messages"])

        if tokens > MAX_FILE_TOKENS:
            raise ValueError(
                f"Task {task['custom_id']} exceeds 2M tokens alone ({tokens})"
            )
        task["_token_count"] = tokens
        tasks.append(task)

In [34]:
import random

random_task = random.choice(tasks)
print(random_task["body"]["messages"][1]["content"])

Sentence need simplify:  "nguyên tắc kinh doanh nhà ở, công trình xây dựng có sẵn bên mua, thuê mua nhà ở, công trình xây dựng, phần diện tích sàn xây dựng trong công trình xây dựng theo quy định của luật này được nhà nước cấp giấy chứng nhận về quyền sử dụng đất, quyền sở hữu tài sản gắn liền với đất đối với nhà ở, công trình xây dựng, phần diện tích sàn xây dựng trong công trình xây dựng đã mua, thuê mua. trình tự, thủ tục, cơ quan nhà nước có thẩm quyền cấp giấy chứng nhận về quyền sử dụng đất, quyền sở hữu tài sản gắn liền với đất thực hiện theo quy định của pháp luật về đất đai."


In [37]:
print(len(tasks))

8686


In [36]:
files = []
current_file = []
current_tokens = 0

for task in tasks:
    if current_tokens + task["_token_count"] > MAX_FILE_TOKENS:
        files.append(current_file)
        current_file = []
        current_tokens = 0

    current_file.append(task)
    current_tokens += task["_token_count"]

if current_file:
    files.append(current_file)

os.makedirs(OUTPUT_PREFIX, exist_ok=True)
for idx, file_tasks in enumerate(files, start=1):
    filename = f"./{OUTPUT_PREFIX}/{OUTPUT_PREFIX}_part{idx}.jsonl"

    with open(filename, "w", encoding="utf-8") as f:
        for task in file_tasks:
            task_copy = dict(task)
            task_copy.pop("_token_count", None)
            task_copy = normalize_for_json(task_copy)
            f.write(json.dumps(task_copy, ensure_ascii=False) + "\n")

    total_tokens = sum(t["_token_count"] for t in file_tasks)

    print(
        f"✔ {filename} | "
        f"{len(file_tasks)} tasks | "
        f"{total_tokens:,} tokens"
    )

print("\n===== SUMMARY =====")
print(f"Total tasks: {len(tasks)}")
print(f"Total files: {len(files)}")
print(f"Total tokens: {sum(t['_token_count'] for t in tasks):,}")

✔ ./batch_property_law_v2/batch_property_law_v2_part1.jsonl | 2762 tasks | 1,999,631 tokens
✔ ./batch_property_law_v2/batch_property_law_v2_part2.jsonl | 2885 tasks | 1,999,531 tokens
✔ ./batch_property_law_v2/batch_property_law_v2_part3.jsonl | 2504 tasks | 1,999,966 tokens
✔ ./batch_property_law_v2/batch_property_law_v2_part4.jsonl | 535 tasks | 403,737 tokens

===== SUMMARY =====
Total tasks: 8686
Total files: 4
Total tokens: 6,402,865


## Run batch api

In [39]:
import os
import time
from openai import OpenAI

gpt_client = OpenAI()

INPUT_SECTIONS_JSONL_FOLDER = r"E:\Github\LawAssistant\src\triplet_extraction\batch_preprocessing\batch_property_law_v2"
RESULT_FOLDER = r"E:\Github\LawAssistant\src\triplet_extraction\batch_preprocessing\batch_property_law_v2\results"

os.makedirs(RESULT_FOLDER, exist_ok=True)

def meta_path(filename):
    return os.path.join(RESULT_FOLDER, f"{filename}.meta.json")

def result_path(filename):
    return os.path.join(RESULT_FOLDER, f"{filename}.result.jsonl")

def process_file(filename):
    input_path = os.path.join(INPUT_SECTIONS_JSONL_FOLDER, filename)
    meta_file = meta_path(filename)
    result_file = result_path(filename)

    if os.path.exists(result_file):
        return

    if os.path.exists(meta_file):
        with open(meta_file, "r", encoding="utf-8") as f:
            meta = json.load(f)
        batch_id = meta["batch_id"]
        batch = gpt_client.batches.retrieve(batch_id)
    else:
        file_obj = gpt_client.files.create(
            file=open(input_path, "rb"),
            purpose="batch"
        )
        batch = gpt_client.batches.create(
            input_file_id=file_obj.id,
            endpoint="/v1/chat/completions",
            completion_window="24h"
        )
        with open(meta_file, "w", encoding="utf-8") as f:
            json.dump(
                {"file_id": file_obj.id, "batch_id": batch.id},
                f,
                indent=2
            )

    while True:
        batch = gpt_client.batches.retrieve(batch.id)
        if batch.status in ("completed", "failed", "expired"):
            break
        print(f"Batch {batch.id} status: {batch.status}. Waiting 5 minutes...")
        time.sleep(300)

    if batch.status != "completed":
        return

    content = gpt_client.files.content(batch.output_file_id)
    with open(result_file, "wb") as f:
        f.write(content.read())

for filename in sorted(os.listdir(INPUT_SECTIONS_JSONL_FOLDER)):
    if not filename.endswith(".jsonl"):
        continue
    try:
        process_file(filename)
    except Exception:
        continue

Batch batch_697a3bb0de3881909cdbeda85638b4af status: validating. Waiting 5 minutes...
Batch batch_697a3bb0de3881909cdbeda85638b4af status: in_progress. Waiting 5 minutes...
Batch batch_697a3bb0de3881909cdbeda85638b4af status: in_progress. Waiting 5 minutes...
Batch batch_697a3bb0de3881909cdbeda85638b4af status: in_progress. Waiting 5 minutes...
Batch batch_697a3bb0de3881909cdbeda85638b4af status: in_progress. Waiting 5 minutes...
Batch batch_697a3bb0de3881909cdbeda85638b4af status: in_progress. Waiting 5 minutes...
Batch batch_697a3bb0de3881909cdbeda85638b4af status: in_progress. Waiting 5 minutes...
Batch batch_697a3bb0de3881909cdbeda85638b4af status: in_progress. Waiting 5 minutes...
Batch batch_697a3bb0de3881909cdbeda85638b4af status: in_progress. Waiting 5 minutes...
Batch batch_697a3bb0de3881909cdbeda85638b4af status: in_progress. Waiting 5 minutes...
Batch batch_697a3bb0de3881909cdbeda85638b4af status: in_progress. Waiting 5 minutes...
Batch batch_697a3bb0de3881909cdbeda85638b4af

## Lưu kết quả xử lý vào MongoDB

In [43]:
from src.db import init_mongo
from pathlib import Path
import json
from bson import ObjectId
from tqdm import tqdm

# Folder containing all batch result files
RESULT_FOLDER = Path(
    r"E:\Github\LawAssistant\src\triplet_extraction\batch_preprocessing\batch_property_law_v2\results"
)

mongo_client = init_mongo()
if not mongo_client:
    print("Failed to connect to MongoDB. Exiting.")
    exit(1)

db = mongo_client["KB_PROPERTY_LAW"]

section_collection = db["legal_sections"]
collection = db["processed_legal_sections"]

# Collect all JSONL result files
files = sorted(RESULT_FOLDER.glob("*.jsonl"))

total_sentences = 0
missing_sections = 0

# File-level progress bar
for file in tqdm(files, desc="Processing result files", unit="file"):
    with open(file, "r", encoding="utf-8") as f:
        lines = f.readlines()

    # Line-level progress bar
    for line in tqdm(lines, desc=file.name, unit="line", leave=False):
        data = json.loads(line)

        if data.get("response", {}).get("status_code") != 200:
            continue

        raw_content = data["response"]["body"]["choices"][0]["message"]["content"].strip()
        try:
            parsed = json.loads(raw_content)
        except json.JSONDecodeError:
            continue

        sentences = parsed.get("simplified_sentences", [])
        if not sentences:
            continue

        section_id = data["custom_id"].split("_part")[0]
        section_doc = section_collection.find_one(
            {"_id": section_id},
            {"so_hieu": 1}
        )

        if not section_doc:
            missing_sections += 1
            continue

        so_hieu = section_doc["so_hieu"]

        sequence = 1
        for sentence in sentences:
            sentence = sentence.strip()
            if not sentence:
                continue

            collection.update_one(
                {
                    "section_id": section_id,
                    "sequence": sequence
                },
                {
                    "$setOnInsert": {
                        "content": sentence,
                        "so_hieu": so_hieu
                    }
                },
                upsert=True
            )

            total_sentences += 1
            sequence += 1

print(f"Sentences inserted : {total_sentences}")
print(f"Missing sections   : {missing_sections}")
print(f"Files processed    : {len(files)}")

You successfully connected to MongoDB!


Processing result files: 100%|██████████| 4/4 [17:33<00:00, 263.47s/file]                          

Sentences inserted : 42999
Missing sections   : 0
Files processed    : 4


## Lưu kết quả xử lý vào SQLite

In [2]:
import json
from tqdm import tqdm
import sqlite3

conn = sqlite3.connect("process_law.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS laws (
    id TEXT,
    sequence INTEGER,
    content TEXT,
    so_hieu TEXT NOT NULL,
    PRIMARY KEY (id, sequence)
);
""")
conn.commit()


In [3]:
# Paths
db_path = r"/src/triplet_extraction/batch_preprocessing/process_law.db"
batch_result_path = r"/src/triplet_extraction\batch_68f0720bbe3081908aa61019a6d518fe_output.jsonl"

# Initialize database connection
conn = sqlite3.connect(db_path)

cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print("Các bảng trong law.db:", tables)

Các bảng trong law.db: [('laws',)]


In [4]:
import sqlite3
gtvt_conn = sqlite3.connect(r"/src/triplet_extraction/batch_preprocessing/GTVT_law.db")
gtvt_cursor = gtvt_conn.cursor()

In [5]:
def extract_from_sqlite(cursor, id: int, include_parent=False):
    cursor.execute("SELECT * FROM laws WHERE id = ?", (id,))
    rows = cursor.fetchall()

    columns = [col[0] for col in cursor.description]
    results = [dict(zip(columns, row)) for row in rows]

    if include_parent:
        all_nodes = []
        for r in results:
            current = r
            while current['parent_id'] is not None:
                cursor.execute("SELECT * FROM laws WHERE id = ?", (current['parent_id'],))
                parent = cursor.fetchone()
                if parent is None:
                    break
                parent_dict = dict(zip(columns, parent))
                all_nodes.append(parent_dict)
                current = parent_dict
        results.extend(all_nodes)

    return results[::-1]

In [6]:
with open(batch_result_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

inserted = 0

for line in tqdm(lines, desc="Inserting JSON results", unit="entry"):
    data = json.loads(line)
    if data.get("response", {}).get("status_code") != 200:
        continue

    content = data["response"]["body"]["choices"][0]["message"]["content"]
    content = content.strip()
    split_content = content.replace("\n", "").split(".")
    law_id = data["custom_id"]

    # Fetch so_hieu from original DB if needed
    law_sentence = extract_from_sqlite(gtvt_cursor, law_id)
    so_hieu = law_sentence[0]["so_hieu"] if law_sentence else "unknown"
    sequence = 1
    for c in split_content:
        c = c.strip()
        if not c:
            continue
        cursor.execute("""
            INSERT OR REPLACE INTO laws (id, sequence, content, so_hieu)
            VALUES (?, ?, ?, ?)
        """, (law_id, sequence, c, so_hieu))
        inserted += 1
        sequence += 1

conn.commit()
print(f"Inserted {inserted} records into 'laws' table")

Inserting JSON results: 100%|██████████| 943/943 [00:00<00:00, 22056.20entry/s]

Inserted 4424 records into 'laws' table


In [7]:
law_sentence = extract_from_sqlite(gtvt_cursor, "3f2f65567a03467937fa6b9f291607028d46f8809afdcbf2303e905cb8ed8ab8")
print(law_sentence)

[{'id': '3f2f65567a03467937fa6b9f291607028d46f8809afdcbf2303e905cb8ed8ab8', 'title': 'Điểm e', 'content': 'Tổng hợp chi phí vận hành công trình trạm trong 1 ca (hoặc 1 ngày) làm việc BẢNG 2.2.A - TỔNG HỢP CHI PHÍ NHÂN CÔNG, MÁY THI CÔNG, VẬT TƯ VẬT LIỆU TRONG CHI PHÍ TRỰC TIẾP VẬN HÀNH CÔNG TRÌNH TRẠM TRONG 1 ĐƠN VỊ THỜI GIAN (CA, NGÀY) STT Mã hiệu Nội dung Đơn vị Khối lượng Giá Thành tiền [1] [2] [3] [4] [5] [6] [7] = [5]x[6] I Nhân công', 'parent_id': '63cbb6d6c4f372508b3868b7ac7e5cf310b41a3c4fa8d4ca4c86b32351ca73a0', 'so_hieu': '41/2021/TT-BGTVT'}]
